#### Domain – Retail
#### focus – Optimize Book RENT

**Business challenge/requirement**

BookRent is the largest online and offline book rental chain in India. TheCompany charges a fixed fee per month plus rental per book. So,the company makes more money when users rent more books. 

You as an ML expert have to model a recommendation engine so that user gets recommendation of books based on the behaviorof similar users. This will ensure that users are renting books based on their taste.

The companyis still unprofitable and is looking to improve both revenue and profit.

**Key issues**

As of now a lot of users return the booksand do not take the new rental.  The rightrecommendation will entice a usersto rent more books

**Considerations**

NONE

**Data volume**

- Approx 1 M  records –fileBX-Book-Ratings.csv and 2 more. But only 10K records will be used

Fields in Data
- user_id: Unique Id of the User
- isbn: International Standard Book Number is a unique numeric commercial book identifier
- rating: the rating given by the user

**Additional information**

-NA

**Business benefits**

Increase in both top line and bottom line as more rentals per user means more revenue and more profit

In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import numpy as np

In [2]:
def load_datasets():
    books_rating_df = pd.read_csv('files\\books\\BX-Book-Ratings.csv',encoding='latin-1')
    books_df = pd.read_csv('files\\books\\BX-Books.csv',encoding='latin-1',low_memory=False)
    users_df = pd.read_csv('files\\books\\BX-Users.csv',encoding='latin-1',low_memory=False, sep=',')
    return books_rating_df,books_df,users_df

In [3]:
if __name__ == "__main__":
    ratings, book, users = load_datasets()

In [ ]:
display(ratings.info())
display(ratings.head())

,user_id,isbn,rating
0,276725,034545104X,0
1,276726,155061224,5
2,276727,446520802,0
3,276729,052165615X,3
4,276729,521795028,6


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 3 columns):
 #   Column   Non-Null Count    Dtype 
---  ------   --------------    ----- 
 0   user_id  1048575 non-null  int64 
 1   isbn     1048575 non-null  object
 2   rating   1048575 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 24.0+ MB


None

In [62]:
#book['year_of_publication'] = book['year_of_publication'].astype('int64')
display(book.info())
display(book.head())
book[~book['year_of_publication'].astype(str).str.isnumeric()]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271379 entries, 0 to 271378
Data columns (total 5 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   isbn                 271379 non-null  object
 1   book_title           271379 non-null  object
 2   book_author          271377 non-null  object
 3   year_of_publication  271379 non-null  object
 4   publisher            271377 non-null  object
dtypes: object(5)
memory usage: 10.4+ MB


None

,isbn,book_title,book_author,year_of_publication,publisher
0,195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press
1,2005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada
2,60973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial
3,374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux
4,393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company


,isbn,book_title,book_author,year_of_publication,publisher
6450,735201994,Peterman Rides Again: Adventures Continue with...,"the Catalog Business""",John Peterman,2000
43665,330482750,"Three Novels: \A Strange and Sublime Address\""","\""Afternoon Raag\""","\""Freedom Song\""""",Amit Chaudhuri
51749,413326608,"Upon my word!: More stories from \My word!\"" a...","Tony Shryane""",Frank Muir,1974
92036,440500702,Schrodinger's Cat Trilogy : \The Universe Next...,"\""The Homing Pigeons\""""",ROBERT A. WILSON,1988
104317,373166982,"Please Say \I Do\"" (Three Weddings &amp","A Hurricane) (Harlequin American Romance, No ...",Karen T. Whittenburg,1997
121766,894805959,The Best of the Journal of Irreproducible Resu...,"Unfounded Findings\""""",George H. Scherr,1989
144056,8423920143,"GuÃ­a del lector del \Quijote\"": Ensayo psicol...","14 : Ensayo)""",Salvador de Madariaga,1976
150787,034050823X,"The Double Detective: \The Blank Page\"" &amp","\""A Fix Like This\"" (Crime Club)""",K.C. Constantine,1990
157126,039482492X,"C is for Clown: A Circus of \C\"" Words, (Brigh...","early books for beginning beginners)""",Stan Berenstain,1972
180187,553570722,"\R\"" for Revenge: Written by Kate William",Created by Francine Pascal (Sweet Valley High...,Francine Pascal,1997


In [58]:
users.info()
display(users.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 278859 entries, 0 to 278858
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   user_id   278859 non-null  object 
 1   Location  278858 non-null  object 
 2   Age       168096 non-null  float64
dtypes: float64(1), object(2)
memory usage: 6.4+ MB


,user_id,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


In [47]:
users['Age'] = users['Age'].fillna(0)
users[users['Location'].isnull()]

,user_id,Location,Age
275081,", milan, italy""",NaN,0.0
